# Day 029 Project Solution — NotificationBot

A `NotificationBot` that converts automation events into AI-generated Slack and Discord notifications.

In [ ]:
import ollama


def format_slack_message(
    title: str,
    body: str,
    color: str = "#36a64f",
) -> dict:
    from datetime import datetime
    return {
        "attachments": [
            {
                "fallback": title,
                "color":    color,
                "title":    title,
                "text":     body,
                "footer":   "NotificationBot",
                "ts":       int(datetime.now().timestamp()),
            }
        ]
    }


def format_discord_embed(
    title: str,
    description: str,
    color: int = 0x00b0f4,
) -> dict:
    return {
        "title":       title,
        "description": description,
        "color":       color,
    }


def truncate_for_chat(text: str, max_chars: int = 2000) -> str:
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."


def ai_summarize_for_chat(
    content: str,
    platform: str = "slack",
    model: str = "llama3.2",
) -> str:
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    f"You are a notification writer for {platform}. "
                    "Write a concise notification summary: under 300 characters, "
                    "no headers, no bullet points. Lead with the most important fact."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Summarise this for a {platform} notification:\n\n"
                    f"{content[:3000]}"
                ),
            },
        ],
    )
    return response["message"]["content"]


def build_notification(
    event_type: str,
    data: dict,
    model: str = "llama3.2",
) -> dict:
    content = (
        f"Event: {event_type}\n\nData:\n"
        + "\n".join(f"  {k}: {v}" for k, v in data.items())
    )
    summary = ai_summarize_for_chat(content, platform="slack", model=model)
    title   = f"[{event_type.upper()}] Notification"
    return {
        "event_type":      event_type,
        "title":           title,
        "summary":         summary,
        "slack_payload":   format_slack_message(title, summary),
        "discord_payload": {
            "embeds": [format_discord_embed(title, truncate_for_chat(summary, 4096))]
        },
    }


class NotificationBot:
    def __init__(
        self,
        slack_url: str | None = None,
        discord_url: str | None = None,
        model: str = "llama3.2",
    ):
        self.slack_url   = slack_url
        self.discord_url = discord_url
        self.model       = model

    def preview(self, event_type: str, data: dict) -> dict:
        return build_notification(event_type, data, model=self.model)

    def notify(self, event_type: str, data: dict) -> dict:
        result = self.preview(event_type, data)
        result["sent"] = []
        if self.slack_url:
            import requests
            r = requests.post(
                self.slack_url, json=result["slack_payload"], timeout=10
            )
            result["sent"].append(f"slack:{r.status_code}")
        if self.discord_url:
            import requests
            r = requests.post(
                self.discord_url, json=result["discord_payload"], timeout=10
            )
            result["sent"].append(f"discord:{r.status_code}")
        return result

## Action 1 — Build Slack and Discord Payloads Directly

In [ ]:
# Slack green success payload
slack_ok = format_slack_message(
    title='\u2705 Report Generated',
    body='Sales report for 5 products completed successfully.',
    color='#36a64f',
)
print('Slack payload keys:', list(slack_ok))
print('Attachment title: ', slack_ok['attachments'][0]['title'])
print('Attachment color: ', slack_ok['attachments'][0]['color'])

# Discord blue info embed
discord_embed = format_discord_embed(
    title='\U0001f4ca Sales Report Ready',
    description='5 products analysed. Epsilon X leads with $3,600 in Q4.',
    color=0x2ecc71,
)
discord_payload = {'embeds': [discord_embed]}
print('\nDiscord embed title:', discord_payload['embeds'][0]['title'])
print('Discord embed color:', discord_payload['embeds'][0]['color'])

# Truncation guard
long_body = 'X' * 3000
safe = truncate_for_chat(long_body, max_chars=2000)
print(f'\nTruncated {len(long_body)} → {len(safe)} chars, ends with: {safe[-5:]!r}')

## Action 2 — Generate AI-Powered Notification

In [ ]:
bot = NotificationBot()

result = bot.preview(
    event_type='report.generated',
    data={
        'rows_analyzed': 5,
        'top_product': 'Epsilon X',
        'total_revenue': 13350,
        'output': '/tmp/day028_report.xlsx',
    },
)
print('Event type:', result['event_type'])
print('Title:     ', result['title'])
print('Summary:   ', result['summary'])

## Action 3 — Inspect Payloads and Verify

In [ ]:
# Verify slack payload
sp = result['slack_payload']
print('Slack attachment title:', sp['attachments'][0]['title'])
print('Slack attachment body: ', sp['attachments'][0]['text'][:80])

# Verify discord payload
dp = result['discord_payload']
print('\nDiscord embeds count:', len(dp['embeds']))
print('Discord embed title: ', dp['embeds'][0]['title'])
print('Discord embed desc:  ', dp['embeds'][0]['description'][:80])

# Payloads would be sent like this (commented — no real webhook URL):
# import requests
# requests.post(SLACK_WEBHOOK, json=sp, timeout=10).raise_for_status()
# requests.post(DISCORD_WEBHOOK, json=dp, timeout=10).raise_for_status()

print('\nNotification bot complete!')